# validation error — Python demo

Numerical companion to the entry [validation error](https://dictionaryofml.org/terms/valerr.html) of the [Dictionary of Applied Machine Learning](https://dictionaryofml.org/): it recomputes what the entry states and prints one line per check.

The entry's weather narrative, carried out on the record itself: the daily minimum and maximum air temperature at Krems an der Donau for all 366 days of 2024 (GeoSphere Austria, station 3805, dataset klima-v2-1d). Each day is one data point: its feature is the morning minimum, its label the maximum of the day.

Requires NumPy and Matplotlib only, and uses fixed seeds, so the printed numbers reproduce exactly. Generated from [`pythondemos/valerr.py`](https://dictionaryofml.org/terms/valerr.py); CC BY 4.0.

In [ ]:
# Notebook shim: the script resolves output paths relative to __file__,
# which a notebook kernel does not define; everything lands in the
# working directory instead.
import os
__file__ = os.path.join(os.getcwd(), "valerr.py")
os.makedirs("pythondemos", exist_ok=True)

In [ ]:
"""
valerr.py -- numerical companion to the entry 'validation error'.

The entry's weather narrative, carried out on the record itself: the daily
minimum and maximum air temperature at Krems an der Donau for all 366 days
of 2024 (GeoSphere Austria, station 3805, dataset klima-v2-1d).  Each day
is one data point: its feature is the morning minimum, its label the
maximum of the day.

The picture the entry opens with uses six days spread over the range of
morning minima: four of them a training set, two held back.  A quadratic
fitted to the four incurs an average squared error of 0.24 on them and of
25.43 on the two held-back days.  The second number is the validation
error, and it is the one that says what the curve is worth on a day it
never saw.

The larger example splits the year: every ninth day, forty days, is the
training set; twenty of the remaining days are the validation set; the
other 306 stay held back and stand in for the risk.  For a straight line
the three numbers are 18.96, 18.59 and 19.41.  The validation error sits
near the risk while the training error is below it, but for a line the
gap is 0.45 and a twenty-day average fluctuates by about 5.4, so one
validation set cannot resolve it.  Raising the degree makes the gap plain:
at degree five the three numbers are 17.93, 20.84 and 21.45, and at degree
nine they are 13.80, 35.12 and 228.96.

Finally the fluctuation itself: the validation error is an average of
per-day losses, so it is a random quantity.  Recomputed over 300 draws of
a validation set of each size from the held-back days, its mean stays near
the risk while its spread falls from 11.36 at five days to 2.26 at eighty,
close to the standard deviation of a single day's loss divided by the
square root of the size, corrected for drawing without replacement.

Deterministic: fixed seed for the draws; every fit is the closed-form
least-squares solution and the subsets are fixed strides through the year.
Self-contained: numpy + matplotlib only (stdlib urllib for the download).

Blocks
------
[B-fetch]   Download the daily minimum and maximum temperature at Krems
            for 2024; write the 366 records to valerr_weather.csv and
            check them against the archive.
[B-picture] The six days of the entry's opening figure: four training
            days, two held back, and the quadratic fitted to the four.
            The validation error is the average over the two held-back
            days; check that it dwarfs the training error, and write the
            points, the curve and the prediction errors for the figure.
[B-def]     The validation error of the line fitted to the forty training
            days: the average of twenty squared errors, with the line
            fixed.
[B-risk]    The validation error against the risk, for degrees 1, 5 and 9:
            the training error is below the risk at every degree, and the
            gap grows with the degree, while for the line it is smaller
            than the fluctuation of a twenty-day average.
[B-spread]  The validation error recomputed over 300 draws of a validation
            set of each size: the spread shrinks as the set grows.  Writes
            the CSV behind the entry's second figure.

Outputs
-------
valerr_weather.csv      : date, tmin, tmax of the 366 downloaded days
valerr_picture_train.csv: tmin, tmax of the four training days
valerr_picture_val.csv  : tmin, tmax of the two held-back days
valerr_picture_curve.csv: tmin, tmax along the fitted quadratic
valerr_picture_err.csv  : tmin, curve, errplus, errminus -- the vertical
                          gap between each day and the curve
valerr_spread.csv       : size, mean, spread of the validation error over
                          300 draws of a validation set of that size
valerr_risk.csv         : size, risk -- the reference level for the figure
valerr.png              : preview (checking only)
"""

import json
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT_DIR = Path(__file__).parent

report = []


def check(name, ok):
    report.append((name, bool(ok)))
    print(f"  [{'ok' if ok else 'FAIL'}] {name}")


def avg_sqerr(x, y, coef):
    return float(np.mean((y - np.polyval(coef, x)) ** 2))

**[B-fetch]** Download the daily minimum and maximum temperature at Krems for 2024; write the 366 records to valerr_weather.csv and check them against the archive.

In [ ]:
print("[B-fetch] the record")
URL = ("https://dataset.api.hub.geosphere.at/v1/station/historical/"
       "klima-v2-1d?parameters=tlmin,tlmax&station_ids=3805"
       "&start=2024-01-01&end=2024-12-31")
with urllib.request.urlopen(URL, timeout=120) as resp:
    payload = json.load(resp)
params = payload["features"][0]["properties"]["parameters"]
dates = [stamp[:10] for stamp in payload["timestamps"]]
tmin = np.array(params["tlmin"]["data"], dtype=float)
tmax = np.array(params["tlmax"]["data"], dtype=float)
with open(OUT_DIR / "valerr_weather.csv", "w") as f:
    f.write("date,tmin,tmax\n")
    for day, lo, hi in zip(dates, tmin, tmax):
        f.write(f"{day},{lo},{hi}\n")
print(f"    {len(dates)} days, morning minimum from {tmin.min():.1f} to "
      f"{tmin.max():.1f}, maximum from {tmax.min():.1f} to {tmax.max():.1f}")
check("[B-fetch] 366 days downloaded for 2024", len(dates) == 366)
check("[B-fetch] the record matches the archive (Jan 1: 2.6 to 9.6)",
      (dates[0], tmin[0], tmax[0]) == ("2024-01-01", 2.6, 9.6))

**[B-picture]** The six days of the entry's opening figure: four training days, two held back, and the quadratic fitted to the four. The validation error is the average over the two held-back days; check that it dwarfs the training error, and write the points, the curve and the prediction errors for the figure.

In [ ]:
print("\n[B-picture] the six days of the entry's opening figure")
by_feature = np.argsort(tmin, kind="stable")
QUANTILES = (0.05, 0.22, 0.40, 0.58, 0.76, 0.95)
six = np.array([by_feature[int(q * (len(tmin) - 1))] for q in QUANTILES])
pic_train, pic_val = six[[0, 2, 3, 5]], six[[1, 4]]   # the 2nd and 5th held back
quad = np.polyfit(tmin[pic_train], tmax[pic_train], 2)
pic_trainerr = avg_sqerr(tmin[pic_train], tmax[pic_train], quad)
pic_valerr = avg_sqerr(tmin[pic_val], tmax[pic_val], quad)
for label, group in (("training", pic_train), ("held back", pic_val)):
    for d in group:
        print(f"    {label:>9}: {dates[d]}  morning minimum {tmin[d]:5.1f}, "
              f"maximum {tmax[d]:5.1f}, curve {np.polyval(quad, tmin[d]):5.1f}")
print(f"    training error {pic_trainerr:.2f} on the four training days, "
      f"validation error {pic_valerr:.2f} on the two held-back days")
check("[B-picture] six days spread over the range of morning minima",
      len(np.unique(six)) == 6 and tmin[six].max() - tmin[six].min() > 20.0)
check("[B-picture] the validation error is more than ten times the training "
      "error", pic_valerr > 10.0 * pic_trainerr)

grid = np.linspace(tmin[six].min() - 1.5, tmin[six].max() + 1.5, 200)
np.savetxt(OUT_DIR / "valerr_picture_curve.csv",
           np.stack([grid, np.polyval(quad, grid)], 1), delimiter=",",
           header="tmin,tmax", comments="", fmt="%.3f")
for name, group in (("train", pic_train), ("val", pic_val)):
    np.savetxt(OUT_DIR / f"valerr_picture_{name}.csv",
               np.stack([tmin[group], tmax[group]], 1), delimiter=",",
               header="tmin,tmax", comments="", fmt="%.2f")
with open(OUT_DIR / "valerr_picture_err.csv", "w") as f:
    f.write("tmin,curve,errplus,errminus\n")
    for d in np.sort(six):
        on_curve = float(np.polyval(quad, tmin[d]))
        gap = float(tmax[d]) - on_curve
        f.write(f"{tmin[d]:.2f},{on_curve:.3f},{max(gap, 0.0):.3f},"
                f"{max(-gap, 0.0):.3f}\n")

**[B-def]** The validation error of the line fitted to the forty training days: the average of twenty squared errors, with the line fixed.

In [ ]:
print("\n[B-def] the validation error of a fitted line")
idx = np.arange(len(tmin))
train = idx[::9][:40]                       # every ninth day, forty of them
rest = np.setdiff1d(idx, train)
val = rest[::16][:20]                       # twenty of the remaining days
held = np.setdiff1d(rest, val)              # the other 306, standing in for the risk
line = np.polyfit(tmin[train], tmax[train], 1)
losses = (tmax[val] - np.polyval(line, tmin[val])) ** 2
valerr = avg_sqerr(tmin[val], tmax[val], line)
print(f"    {len(train)} training days, {len(val)} validation days, "
      f"{len(held)} held back")
print(f"    validation error {valerr:.2f} = average of {len(val)} squared "
      f"errors (smallest {losses.min():.2f}, largest {losses.max():.2f})")
check("[B-def] the validation error is the average of the per-day losses",
      np.isclose(valerr, float(np.mean(losses)), atol=1e-12))
check("[B-def] the line is fixed: computing the validation error changed no "
      "coefficient", np.allclose(line, np.polyfit(tmin[train], tmax[train], 1)))
check("[B-def] no validation day is a training day", not set(val) & set(train))

**[B-risk]** The validation error against the risk, for degrees 1, 5 and 9: the training error is below the risk at every degree, and the gap grows with the degree, while for the line it is smaller than the fluctuation of a twenty-day average.

In [ ]:
print("\n[B-risk] the two errors against the risk")
three = []
for degree in (1, 5, 9):
    coef = np.polyfit(tmin[train], tmax[train], degree)
    three.append((degree, avg_sqerr(tmin[train], tmax[train], coef),
                  avg_sqerr(tmin[val], tmax[val], coef),
                  avg_sqerr(tmin[held], tmax[held], coef)))
    print(f"    degree {degree}: training error {three[-1][1]:6.2f}, validation "
          f"error {three[-1][2]:6.2f}, risk {three[-1][3]:8.2f}")
per_day = (tmax[held] - np.polyval(line, tmin[held])) ** 2
noise = float(per_day.std()) / np.sqrt(len(val))
print(f"    for the line the training error is below the risk by "
      f"{three[0][3] - three[0][1]:.2f}, while a {len(val)}-day average "
      f"fluctuates by about {noise:.2f}")
check("[B-risk] the training error is below the risk at every degree",
      all(t < r for _, t, _, r in three))
check("[B-risk] the gap between the training error and the risk grows with "
      "the degree",
      all(a[3] - a[1] < b[3] - b[1] for a, b in zip(three, three[1:])))
check("[B-risk] for the line that gap is smaller than the fluctuation of a "
      "twenty-day average", three[0][3] - three[0][1] < noise)

**[B-spread]** The validation error recomputed over 300 draws of a validation set of each size: the spread shrinks as the set grows. Writes the CSV behind the entry's second figure.

In [ ]:
print("\n[B-spread] the fluctuation of the validation error")
SIZES = [5, 10, 20, 40, 80]
DRAWS = 300
risk = float(per_day.mean())
gen = np.random.default_rng(0)
rows = []
for size in SIZES:
    means = [float(per_day[gen.choice(len(per_day), size, replace=False)].mean())
             for _ in range(DRAWS)]
    rows.append((size, float(np.mean(means)), float(np.std(means))))
    # the days are drawn without replacement from a finite pool, so the
    # spread of the average carries the finite-population correction
    expected = (per_day.std() / np.sqrt(size)
                * np.sqrt((len(per_day) - size) / (len(per_day) - 1)))
    print(f"    {size:>2} validation days: mean {rows[-1][1]:6.2f}, spread "
          f"{rows[-1][2]:5.2f} (a single day's loss has spread "
          f"{per_day.std():.2f}, so {expected:5.2f} is expected)")
check("[B-spread] the spread shrinks as the validation set grows",
      all(a[2] > b[2] for a, b in zip(rows, rows[1:])))
check("[B-spread] each mean stays within 10% of the risk",
      all(abs(m - risk) < 0.10 * risk for _, m, _ in rows))
def expected_spread(n):
    """The spread of an average of n of the held-back losses, drawn without
    replacement: one day's spread over the square root of n, times the
    finite-population correction."""
    return (per_day.std() / np.sqrt(n)
            * np.sqrt((len(per_day) - n) / (len(per_day) - 1)))


check("[B-spread] each spread is within 10% of the standard deviation of a "
      "single day's loss divided by the square root of the size, corrected "
      "for drawing without replacement",
      all(abs(s - expected_spread(n)) < 0.10 * expected_spread(n)
          for n, _, s in rows))

with open(OUT_DIR / "valerr_spread.csv", "w") as fh:
    fh.write("size,mean,spread\n")
    for n, m, s in rows:
        fh.write(f"{n},{m:.4f},{s:.4f}\n")
with open(OUT_DIR / "valerr_risk.csv", "w") as fh:
    fh.write("size,risk\n")
    fh.write(f"{SIZES[0]},{risk:.4f}\n")
    fh.write(f"{SIZES[-1]},{risk:.4f}\n")


# ---- preview (checking only)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
ax = axes[0]
ax.plot(grid, np.polyval(quad, grid), "k-", lw=1.4, label="fitted quadratic")
for d in np.sort(six):
    ax.plot([tmin[d], tmin[d]], [np.polyval(quad, tmin[d]), tmax[d]], "k:", lw=1.0)
ax.plot(tmin[pic_train], tmax[pic_train], "o", color="black", ms=7,
        label="training set (4 days)")
ax.plot(tmin[pic_val], tmax[pic_val], "^", mfc="none", mec="black", ms=9,
        mew=1.5, label="validation set (2 days)")
ax.set_xlabel("morning minimum temperature (feature, °C)")
ax.set_ylabel("maximum temperature (label, °C)")
ax.set_title(f"Six days at Krems: training error {pic_trainerr:.2f}, "
             f"validation error {pic_valerr:.2f}", fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax = axes[1]
ax.errorbar([r[0] for r in rows], [r[1] for r in rows],
            yerr=[r[2] for r in rows], fmt="o", ms=5, color="black",
            capsize=3, label="validation error (mean and spread)")
ax.axhline(risk, ls="--", color="0.4", label="risk of the fixed line")
ax.set_xscale("log")
ax.set_xticks(SIZES)
ax.set_xticklabels([str(s) for s in SIZES])
ax.set_xlabel("number of validation days")
ax.set_ylabel("validation error")
ax.set_title("The average fluctuates less as the validation set grows",
             fontsize=9)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(OUT_DIR / "valerr.png", dpi=150)
plt.close(fig)

passed = sum(1 for _, ok in report if ok)
print(f"\n{passed}/{len(report)} checks pass")